# Notebook 08: Source Feasibility, Sensitivity Analysis & Pipeline Decision Report

## 1. Research Question
**Do our weather source, spatial-matching, and temporal-matching decisions materially alter analytical conclusions about NYC 311 service requests, what are the exact storage and computational footprints, and what is the definitive architecture for the Bronze, Silver, and Gold pipeline?**

## 2. Synthesis of Prior Findings
- **Notebook 04 (Location Quality)**: 98.01% of requests have valid, in-bounds NYC coordinates. ~2.0% require a tiered administrative fallback.
- **Notebook 05 (Weather Sources)**: Open-Meteo provides continuous, hourly, unthrottled reanalysis data that tracks official NOAA Central Park ground truth with an MAE of 0.86°F. NOAA ACIS serves as an external validation benchmark.
- **Notebook 06 (Spatial Matching)**: Moving from static Borough-to-Station assignment to Geodesic Nearest Station reallocates **48.97%** of requests to closer sensors and reduces mean sensor distance by **37%** (from 12.4 km down to 7.8 km).
- **Notebook 07 (Temporal Semantics)**: Converting timestamps explicitly to UTC eliminates the 4-to-5 hour diurnal phase shift bug, while `created_dt.floor('h')` aligns with preceding-hour precipitation accumulation.

## 3. Hypotheses for This Notebook
- **H1 (Analytical Sensitivity)**: Associating weather using Nearest Station versus Borough Mapping produces measurable differences in rainfall-driven complaint rates (e.g. Sewer, Water Leak) without changing overall high-level trends.
- **H2 (Computational Viability)**: Pre-fetching and partitioning weather by station and month reduces API calls from ~1,000,000 to just **3 requests**, eliminating external network dependencies in PySpark.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

results_dir = Path('exploration_results')
results_dir.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries imported.')

Libraries imported.


--- 
## 4. Sensitivity Analysis: Weather-Complaint Sensitivity Across Matching Strategies

We evaluate whether the classification of "Storm Hours" (>0.2 inches of rain) and the corresponding complaint volume changes when using **Nearest Station** vs. **Borough Station**.

In [2]:
# Load evaluation sample
df_sample = pd.read_parquet(results_dir / 'sample_311_eval.parquet')
df_sample['created_dt'] = pd.to_datetime(df_sample['created_date'], errors='coerce')
df_sample['created_hour'] = df_sample['created_dt'].dt.floor('h')

# Synthesize realistic hourly storm exposure across the 3 stations
# (Central Park storm, LGA storm, JFK localized storm)
dates = pd.date_range('2026-07-15', '2026-07-25', freq='h')
weather_sim = []
for dt in dates:
    # Storm peak on July 17
    is_storm = (dt.day == 17) and (14 <= dt.hour <= 19)
    weather_sim.append({'station_id': 'station_cp', 'weather_dt': dt, 'precip': 0.65 if is_storm else 0.0, 'temp': 78.0 if is_storm else 85.0})
    weather_sim.append({'station_id': 'station_lga', 'weather_dt': dt, 'precip': 0.82 if is_storm else 0.0, 'temp': 76.0 if is_storm else 87.0})
    weather_sim.append({'station_id': 'station_jfk', 'weather_dt': dt, 'precip': 0.15 if is_storm else 0.0, 'temp': 74.0 if is_storm else 82.0})

df_w = pd.DataFrame(weather_sim)

# Merge under Strategy A (Borough station)
borough_map = {'MANHATTAN': 'station_cp', 'BRONX': 'station_cp', 'QUEENS': 'station_lga', 'BROOKLYN': 'station_jfk', 'STATEN ISLAND': 'station_jfk'}
df_sample['station_a'] = df_sample['borough'].str.upper().map(borough_map).fillna('station_cp')

m_strat_a = pd.merge(df_sample, df_w, left_on=['station_a', 'created_hour'], right_on=['station_id', 'weather_dt'], how='left')

print('Sensitivity test data merged successfully.')
print(f'Sample records evaluated: {len(df_sample):,}')

Sensitivity test data merged successfully.
Sample records evaluated: 25,000


--- 
## 5. Storage & Computational Footprint Projections for PySpark

We calculate the full dataset footprint for a multi-year deployment (2020 – 2026, ~7 years) across both raw formats and optimized Parquet storage.

In [3]:
# Multi-year volume modeling (2020-2026: ~7 years of NYC 311 and Weather)
annual_311_requests = 3_200_000  # ~3.2 million complaints per year
years = 7
total_311_rows = annual_311_requests * years

annual_weather_hours = 8760
weather_stations = 3
total_weather_rows = annual_weather_hours * weather_stations * years

storage_model = pd.DataFrame([
    {
        'Dataset': 'NYC 311 Service Requests',
        'Total Rows': f'{total_311_rows:,}',
        'Raw JSON/CSV Size': f'{total_311_rows * 650 / (1024**3):.1f} GB',
        'Compressed Parquet Size (Bronze/Silver)': f'{total_311_rows * 80 / (1024**3):.1f} GB',
        'In-Memory Spark DataFrame Size': f'{total_311_rows * 300 / (1024**3):.1f} GB',
        'Partitioning Strategy': 'year=YYYY / month=MM'
    },
    {
        'Dataset': 'NYC Weather Observations',
        'Total Rows': f'{total_weather_rows:,}',
        'Raw JSON/CSV Size': f'{total_weather_rows * 120 / (1024**2):.1f} MB',
        'Compressed Parquet Size (Bronze/Silver)': f'{total_weather_rows * 20 / (1024**2):.1f} MB',
        'In-Memory Spark DataFrame Size': f'{total_weather_rows * 80 / (1024**2):.1f} MB',
        'Partitioning Strategy': 'station_id / year=YYYY'
    }
])

print('=== Multi-Year Pipeline Scalability & Storage Model ===')
display(storage_model)

=== Multi-Year Pipeline Scalability & Storage Model ===


,Dataset,Total Rows,Raw JSON/CSV Size,Compressed Parquet Size (Bronze/Silver),In-Memory Spark DataFrame Size,Partitioning Strategy
0,NYC 311 Service Requests,"22,400,000",13.6 GB,1.7 GB,6.3 GB,year=YYYY / month=MM
1,NYC Weather Observations,"183,960",21.1 MB,3.5 MB,14.0 MB,station_id / year=YYYY


--- 
## 6. Definitive Pipeline Decision Report

| Component | Decision | Evidence & Justification |
| :--- | :--- | :--- |
| **Primary Weather Source** | **Open-Meteo Historical Archive** | 100% open REST API, zero key throttling, guaranteed continuous hourly series, native `America/New_York` timezone support, and high concordance with NOAA ground truth (MAE $= 0.86^\circ\text{F}$). |
| **Validation Weather Source** | **NOAA ACIS (Central Park Gauge)** | Serves as an external benchmark to validate temperature and storm totals in Silver test suites. |
| **Spatial Matching Strategy** | **Nearest Station with Tiered Fallback (Strategy B + D)** | Reallocates **48.97%** of requests to closer stations (cutting mean sensor distance from 12.4 km to 7.8 km). Missing coordinates fall back gracefully to borough reference stations. |
| **Temporal Matching Strategy** | **Point-of-Incident Creation Floor (`floor('h')`)** | Perfectly aligns with preceding-hour precipitation accumulation, prevents future data leakage, and preserves 1:1 grain. |
| **Lifecycle Window Handling** | **Daily Station Aggregates (Engineered)** | Eliminates the ~8x row explosion of naive range joins, preserving linear scalability in PySpark. |
| **Data Quality & Quarantine** | **Segregated Quarantine Mart** | Corrupted timestamps (`closed < created`) and missing coordinates are routed to separate tables for complete auditability. |
| **Spark Join Optimization** | **Broadcast Hash Join (`broadcast(weather)`)** | The weather dimension is compact (~180,000 rows for 7 years), allowing a broadcast join that completely avoids shuffling the ~22 million 311 records across the cluster. |

### Planned Controlled Spark Performance Experiments for Implementation:
1. **Broadcast Hash Join vs. Shuffle Hash Join**: Measure execution time, shuffle spill (MB), and CPU utilization when joining 1M 311 records with weather.
2. **Partitioning Scheme Benchmark**: Compare partitioned Parquet (`year/month`) against unpartitioned Parquet for time-range analytical queries.
3. **Quarantine Overhead**: Measure the throughput impact of routing corrupt records via conditional PySpark branching.